In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!



In [1]:
df = spark.sql("SELECT * FROM LH_MedallionArchitecture.Silver.churn LIMIT 1000")
display(df)

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 3, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, f23c8671-06e9-4e0e-a87f-3a442a496e96)

In [2]:
df = spark.sql("SELECT * FROM LH_MedallionArchitecture.Silver.customers LIMIT 5")
display(df)

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 4, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 54c0de7e-2592-4c39-859c-37d1efdf81c3)

In [3]:
df = spark.sql("SELECT * FROM LH_MedallionArchitecture.Silver.orders LIMIT 10")
display(df)

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 5, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, ed132412-8ba1-486b-a8ca-e93b36900aa0)

In [4]:
df = spark.sql("SELECT * FROM LH_MedallionArchitecture.Silver.products LIMIT 10")
display(df)

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 6, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 5aa50c66-5c10-4c17-9d95-babd23abf14f)

In [5]:
df = spark.sql("SELECT * FROM LH_MedallionArchitecture.Silver.sales LIMIT 1000")
display(df)

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 4019187f-dd31-47ef-96d1-0bed6ffe1091)

In [6]:
spark.read.table("Silver.customers").write.format("delta").mode("overwrite").saveAsTable("Gold.customers")

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 8, Finished, Available, Finished)

In [7]:
spark.read.table("Silver.orders").write.format("delta").mode("overwrite").saveAsTable("Gold.orders")
spark.read.table("Silver.products").write.format("delta").mode("overwrite").saveAsTable("Gold.products")
spark.read.table("Silver.sales").write.format("delta").mode("overwrite").saveAsTable("Gold.sales")
spark.read.table("Silver.churn").write.format("delta").mode("overwrite").saveAsTable("Gold.churn")

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 9, Finished, Available, Finished)

# **Calendar Table**

In [9]:
from pyspark.sql.functions import (
    min, max, sequence, explode, to_date, expr,
    year, month, dayofmonth, weekofyear,
    quarter, date_format, dayofweek
)

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 11, Finished, Available, Finished)

In [10]:
bounds = spark.read.table("silver.orders").select(
    min("OrderDate").alias("min_date"),
    max("OrderDate").alias("max_date")
).collect()[0]

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 12, Finished, Available, Finished)

In [14]:
from pyspark.sql.functions import (
    explode, sequence, to_date, col, expr, 
    year, month, dayofmonth, weekofyear, 
    quarter, date_format, dayofweek
)

calendar_df = (
    spark.createDataFrame([(bounds[0], bounds[1])], ["start_date", "end_date"])
    .select(
        explode(
            sequence(
                to_date(col("start_date")), 
                to_date(col("end_date")), 
                expr("INTERVAL 1 DAY")
            )
        ).alias("date")
    )
    .withColumn("year", year("date"))
    .withColumn("month", month("date"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("week_of_year", weekofyear("date"))
    .withColumn("quarter", quarter("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("day_name", date_format("date", "EEEE"))
    .withColumn("day_of_week", dayofweek("date"))
    .withColumn("year_month", date_format("date", "yyyy-MM"))
)

calendar_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.calendar")

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 16, Finished, Available, Finished)

In [15]:
display(calendar_df)

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 17, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 86c4c652-6a08-47e0-abca-761118e739b7)

In [17]:
calendar_df.write.format("delta").mode("overwrite").saveAsTable("Gold.calendar")

StatementMeta(, 7bdbc3e9-9d31-4d6e-ba0d-16efd17a2934, 19, Finished, Available, Finished)